In [ ]:
import torchvision as thv

from common.nn.dataset.nn_dataset import NNDataset

from common.nn.enum.nets import Nets
from common.nn.enum.losses import Losses
from common.nn.enum.devices import Devices

from common.nn.params.nn_run import NNRun
from common.nn.params.nn_params import NNParams
from common.nn.params.nn_train_params import NNTrainParams
from common.nn.params.nn_model_params import NNModelParams

from common.nn.nn_model import NNModel

from common.utils import Utils
from common.vis_utils import VisUtils

In [ ]:
DS_MEAN : float = 0.1307
DS_STD  : float = 0.3081

ds = NNDataset(
    ds_class=thv.datasets.MNIST
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD)
        ]
    )
)

print(
    ds.input_dim
    , ds.output_dim
    , len(ds.train_loader.dataset)
    , len(ds.val_loader.dataset)
    , len(ds.test_loader.dataset)
)

In [ ]:
n_epochs = 250
optims = ["adam"]

dropout_probs = [0.5]
hidden_dimss = [
    []
    , [500]
    , [1000]
    , [500, 1000]
]

models = [
    model
        for model in [
            NNModel(
                params=NNModelParams(
                    net=Nets.FEED_FWD
                    , device=Devices.CPU
                    , loss=Losses.CROSS_ENTROPY
                )
                , net_params=NNParams(
                    dropout_prob=dropout_prob
                    , hidden_dims=hidden_dims
                    , input_dim=ds.input_dim
                    , output_dim=ds.output_dim
                )
            )
                for dropout_prob in dropout_probs
                for hidden_dims in hidden_dimss
        ]
]

train_params = [
    train_param
        for train_param in [
            NNTrainParams(n_epochs=n_epochs)
                .with_train_loader(value=ds.train_loader)
                .with_val_loader(value=ds.val_loader)
        ]
]

runs = [
    run for run in [
        model.train(params=train_param)
            for model in models
            for train_param in train_params
    ]
]

In [ ]:
N_SAMPLES = 5_000

for checkpoint in NNRun.load("best").checkpoints()[:2]:
    VisUtils.visualize_checkpoint_logits_2d_tsne(checkpoint=checkpoint, ds=ds, n_samples=N_SAMPLES)

In [ ]:
top_runs = [
    run for run in sorted(
        runs
        , key=lambda run: min(
            run.idps, key=lambda idp: idp.val_edp.error
        ).val_edp.error
    )[:3]
]

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=25
    , fig_size=(25, 20)
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Losses"
    , x=[idp.iter_idx for idp in top_runs[0].idps]
    , yss_legend=[[f"{loss_type} of {run.id}" for loss_type in ["training", "validation"]] for run in top_runs]
    , yss=[[[idp.train_edp.loss for idp in run.idps], [idp.val_edp.loss for idp in run.idps]] for run in top_runs]
)

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=25
    , fig_size=(25, 20)
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Errors"
    , x=[idp.iter_idx for idp in top_runs[0].idps]
    , yss_legend=[[f"{err_type} of {run.id}" for err_type in ["training", "validation"]] for run in top_runs]
    , yss=[[[idp.train_edp.error for idp in run.idps], [idp.val_edp.error for idp in run.idps]] for run in top_runs]
)

In [ ]:
print(f"best model is {top_runs[0].id} which achieves validation error of {min(top_runs[0].idps, key=lambda idp: idp.val_edp.error).val_edp.error:.4f}")